# Run independent specialists in parallel

This lab uses `asyncio.gather` to start two independent AgentScope replies together. The specialists receive the same case but do not exchange findings.

![Parallel specialist workflow](figures/parallel-specialists-workflow.svg)

The network and evidence paths begin from the same case and end with separate findings. Python displays both outputs; it does not create a third synthesis agent.

## Step 1: Import AgentScope, concurrency, and timing helpers

This cell imports `asyncio` for concurrent scheduling, `time` for a simple elapsed-time measurement, and the AgentScope classes used in the lab.

In [ ]:
import asyncio
import os
import time

from dotenv import load_dotenv
from agentscope.agent import Agent, ReActConfig
from agentscope.credential import OpenAICredential
from agentscope.message import Msg, TextBlock
from agentscope.model import OpenAIChatModel


## Step 2: Configure the shared model connection

Both specialists use this connection. Whether the model service processes their requests simultaneously depends on the service and available hardware.

In [ ]:
load_dotenv()
model_name = os.getenv("MODEL")
base_url = os.getenv("OLLAMA_BASE_URL")

if not model_name or not base_url:
    raise RuntimeError("Set MODEL and OLLAMA_BASE_URL in .env before running this notebook.")

model = OpenAIChatModel(
    credential=OpenAICredential(api_key="ollama", base_url=base_url),
    model=model_name,
    stream=False,
    parameters=OpenAIChatModel.Parameters(temperature=0, max_tokens=160),
)


## Step 3: Create independent specialists

These agents have different roles but neither needs the other’s output. That independence is what makes parallel scheduling appropriate.

In [ ]:
network_specialist = Agent(
    name="network_specialist",
    system_prompt=(
        "You are the network specialist for a practice security case. "
        "Report what the network alert observed and what the alert cannot show."
    ),
    model=model,
    react_config=ReActConfig(max_iters=2),
)

evidence_specialist = Agent(
    name="evidence_specialist",
    system_prompt=(
        "You are the evidence specialist for a practice security case. "
        "List supported facts, missing evidence, and one next item to collect."
    ),
    model=model,
    react_config=ReActConfig(max_iters=2),
)


## Step 4: Define the shared practice case

Both agents receive the same original facts. The helper creates separate request messages so each agent has its own clearly named input.

In [ ]:
case_materials = """
Practice case INC-204
- Network-monitoring alert: at 09:14 UTC, an automated sensor recorded a workstation making forty-three outbound contacts to 192.0.2.44.
- The local practice list marks 192.0.2.44 as suspicious and says this requires analyst review; it is not proof of malicious activity.
- The alert does not identify the process, user action, payload, or destination port/service for the contacts.
""".strip()

def make_request(question: str) -> Msg:
    """Build one independent analyst request containing the shared case."""
    return Msg(
        name="analyst",
        role="user",
        content=[TextBlock(text=f"Practice case:\n{case_materials}\n\nQuestion: {question}")],
    )

network_request = make_request("What did the alert observe, and what can it not show?")
evidence_request = make_request("What facts are supported, what is missing, and what should we collect next?")


## Step 5: Start both specialist calls together

`asyncio.gather` schedules both `reply()` calls before waiting for the pair of responses. `perf_counter` measures elapsed wall-clock time, which may or may not improve with a local model server.

In [ ]:
started_at = time.perf_counter()
network_response, evidence_response = await asyncio.gather(
    network_specialist.reply(network_request),
    evidence_specialist.reply(evidence_request),
)
elapsed_seconds = time.perf_counter() - started_at

print(f"Both calls completed in {elapsed_seconds:.2f} seconds.")


## Step 6: Display the separate findings

This cell prints the two responses under labels. It compares outputs but does not combine them into a new agent-generated conclusion.

In [ ]:
def response_text(response: Msg) -> str:
    """Extract readable text from an AgentScope response message."""
    return "".join(block.text for block in response.content if isinstance(block, TextBlock))

print("NETWORK FINDING:")
print(response_text(network_response))
print("\nEVIDENCE FINDING:")
print(response_text(evidence_response))


## Step 7: Checkpoint

Replace `asyncio.gather(...)` with two separate `await` statements, one after the other. Compare the elapsed time, then restore the parallel version. The specialist answers should remain independent in either version.